# 📓 Notebook 1: What is a Write-Ahead Log?

**The big question:** *"How do databases not lose your data when the power dies?"*

Imagine a simple key-value store that lives in memory: a Python dict. It is fast, but the moment the process crashes everything is gone. If we save the dict to disk on every write, we are slow — and worse, we can crash *halfway* through saving.

The trick is the **Write-Ahead Log (WAL)**: before changing anything, append a single line to a log file describing what we are about to do. Only then do we mutate state. If we crash, on restart we *replay* the log to rebuild the state.

In this notebook we compare:

1. 🟥 **BAD** — pure in-memory store: fast but loses everything on crash.
2. 🟨 **BETTER** — rewrite the whole state to disk on every write: slow and unsafe mid-write.
3. 🟩 **BEST** — a tiny WAL: append-only, durable, recoverable.

## Learning objectives
- Feel why naïve "save the whole file" is a bad idea.
- Implement a minimal append-only log.
- Replay a log to rebuild state.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/write-ahead-log
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). Reload the window if it doesn't appear.

In [ ]:
import os, json, tempfile, shutil

WORKDIR = tempfile.mkdtemp(prefix="wal_lab_")
print("workdir:", WORKDIR)

## 🟥 Approach 1 (BAD): pure in-memory KV store

Fast as can be, but everything is gone if the process dies.

In [ ]:
class MemoryKV:
    def __init__(self):
        self.data = {}
    def put(self, k, v): self.data[k] = v
    def get(self, k):    return self.data.get(k)

kv = MemoryKV()
kv.put("user:1", "alice")
kv.put("user:2", "bob")
print(kv.get("user:1"))
# imagine the process crashes here -> everything lost

## 🟨 Approach 2 (BETTER but slow + unsafe): rewrite the whole file every write

This survives a clean shutdown but has two problems:

1. Every write rewrites the entire file → O(n) per write.
2. If we crash *while writing*, the file is half-written and the data is corrupt.

In [ ]:
class WholeFileKV:
    def __init__(self, path):
        self.path = path
        self.data = {}
        if os.path.exists(path):
            with open(path) as f:
                self.data = json.load(f)

    def put(self, k, v):
        self.data[k] = v
        with open(self.path, "w") as f:
            json.dump(self.data, f)        # 😬 not atomic — crash here = corrupt
            f.flush(); os.fsync(f.fileno())

    def get(self, k): return self.data.get(k)

path = os.path.join(WORKDIR, "kv.json")
kv = WholeFileKV(path)
kv.put("user:1", "alice")
kv.put("user:2", "bob")
print("on disk:", open(path).read())

## 🟩 Approach 3 (BEST): Write-Ahead Log

The WAL is just a text file we **append** to. Each line is one operation:

```
{"op": "put", "k": "user:1", "v": "alice"}
{"op": "put", "k": "user:2", "v": "bob"}
{"op": "del", "k": "user:1"}
```

Properties that come for free:

- **Append-only writes** → O(1) per write, friendly to disks.
- **Append-only** → nothing already written is ever rewritten, so a crash can only damage the *tail*.

> ⚠️ People often say newline-delimited JSON is "atomic per line". It is not. We will
> break it on purpose in §4 and then fix it properly.
- **Recovery** = read the log start to finish and re-apply each operation.

In [ ]:
class WalKV:
    def __init__(self, log_path):
        self.log_path = log_path
        self.data = {}
        self._replay()                     # rebuild state on startup
        self._log = open(log_path, "a")    # then open in append mode

    def _replay(self):
        if not os.path.exists(self.log_path):
            return
        with open(self.log_path) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    # ⚠️ SKIP-AND-CONTINUE. Looks reasonable, is subtly wrong — see §4.
                    print("  ⚠️ ignoring torn log line:", line[:40])
                    continue
                if rec["op"] == "put":
                    self.data[rec["k"]] = rec["v"]
                elif rec["op"] == "del":
                    self.data.pop(rec["k"], None)

    def _append(self, rec):
        self._log.write(json.dumps(rec) + "\n")
        self._log.flush()
        os.fsync(self._log.fileno())       # force the OS to write to disk

    def put(self, k, v):
        self._append({"op": "put", "k": k, "v": v})
        self.data[k] = v

    def delete(self, k):
        self._append({"op": "del", "k": k})
        self.data.pop(k, None)

    def get(self, k): return self.data.get(k)

    def close(self): self._log.close()

log_path = os.path.join(WORKDIR, "wal.log")
kv = WalKV(log_path)
kv.put("user:1", "alice")
kv.put("user:2", "bob")
kv.delete("user:1")
kv.close()

print("log on disk:")
print(open(log_path).read())

In [ ]:
# Now simulate a restart by creating a new instance pointing at the same log.
kv2 = WalKV(log_path)
print("recovered:", kv2.data)
kv2.close()

## 4. 🔬 Torn writes: where newline-delimited JSON actually breaks

We claimed a crash mid-line is harmless. Let's try to break that claim, twice.

**Attempt 1 — a torn line at the very end.** This is the easy case and the WAL survives it.

In [ ]:
# Append a broken (half-written) line, as if the process died mid-write.
with open(log_path, "a") as f:
    f.write('{"op": "put", "k": "user:3", "v": "char')   # no closing brace, no newline

kv3 = WalKV(log_path)
print("recovered after torn tail:", kv3.data)
kv3.close()
assert kv3.data == {"user:2": "bob"}, kv3.data   # user:1 was deleted earlier
print("✅ torn tail alone: no harm done")

**Attempt 2 — the process restarts and keeps appending.** This is what actually happens:
the WAL is opened in append mode, so the *next* record is written directly onto the end of
the half-finished one. There is no newline between them, so the two records fuse into a
single unparseable line — and `skip-and-continue` throws **both** away.

Worse, everything written *after* that is still replayed. The recovered state is no longer a
**prefix** of what the application wrote: there is a hole in the middle. A store that silently
drops record #4 but keeps #5 and #6 is more dangerous than one that stops at #3, because
nothing downstream can tell.

In [ ]:
kv4 = WalKV(log_path)        # opens in "a" mode, right after the torn bytes
kv4.put("user:4", "dana")    # this record gets glued onto the torn one
kv4.put("user:5", "erin")    # this one is fine
kv4.close()

kv5 = WalKV(log_path)
print("recovered:", kv5.data)
kv5.close()

# user:4 vanished, but user:5 — written LATER — survived. That is a hole, not a prefix.
assert "user:4" not in kv5.data, "expected the glued record to be lost"
assert kv5.data.get("user:5") == "erin"
print("\n💥 recovered state is NOT a prefix of the writes: user:4 is missing, user:5 is present")

## 5. 🟩 The real fix: framed records + truncate-at-first-bad

Production WALs (Postgres, RocksDB, Kafka, etcd) do not delimit records with newlines. Each
record is **length-prefixed and checksummed**:

```
┌──────────────┬──────────────┬───────────────────────┐
│ length (4 B) │  crc32 (4 B) │  payload (length B)   │
└──────────────┴──────────────┴───────────────────────┘
```

That buys two things a newline cannot:

1. **You know where the record ends before you read it.** A short read means the tail is torn.
2. **You know whether the bytes are the ones you wrote.** A length that survived the crash but a
   payload that did not still fails the CRC. (The `checksum` lab in this folder goes deeper on
   what a CRC does and does not protect against.)

And the recovery rule changes from *skip* to **truncate**: at the first record that does not
verify, stop, cut the file back to the end of the last good record, and continue from there.
The log is then a clean prefix again — which is exactly the guarantee the rest of the database
is built on.

In [ ]:
import struct, zlib

HEADER = struct.Struct(">II")   # length, crc32

class FramedWal:
    """Append-only log of [len][crc32][payload] records with truncating recovery."""

    def __init__(self, path):
        self.path = path
        self.data = {}
        self.truncated_bytes = 0
        self._recover()
        self._log = open(path, "ab")

    # ---- recovery: replay until the first bad record, then TRUNCATE ----
    def _recover(self):
        if not os.path.exists(self.path):
            return
        good_end = 0
        with open(self.path, "rb") as f:
            while True:
                hdr = f.read(HEADER.size)
                if len(hdr) < HEADER.size:
                    break                                   # torn header -> stop
                n, crc = HEADER.unpack(hdr)
                payload = f.read(n)
                if len(payload) < n:
                    break                                   # torn payload -> stop
                if zlib.crc32(payload) != crc:
                    break                                   # corrupt bytes -> stop
                rec = json.loads(payload)
                if rec["op"] == "put":
                    self.data[rec["k"]] = rec["v"]
                elif rec["op"] == "del":
                    self.data.pop(rec["k"], None)
                good_end = f.tell()                         # end of the last GOOD record

        # Cut the garbage tail off so the next append starts from a clean boundary.
        size = os.path.getsize(self.path)
        if good_end < size:
            self.truncated_bytes = size - good_end
            with open(self.path, "r+b") as f:
                f.truncate(good_end)
            print(f"  ✂️  truncated {self.truncated_bytes} unreadable bytes from the tail")

    def _append(self, rec):
        payload = json.dumps(rec).encode()
        self._log.write(HEADER.pack(len(payload), zlib.crc32(payload)) + payload)
        self._log.flush()
        os.fsync(self._log.fileno())

    def put(self, k, v):
        self._append({"op": "put", "k": k, "v": v}); self.data[k] = v

    def delete(self, k):
        self._append({"op": "del", "k": k}); self.data.pop(k, None)

    def close(self):
        self._log.close()

### Same torture test, framed log

We replay the exact sequence that broke the newline WAL: write, crash mid-record, restart,
keep writing. The invariant we check is the one that matters — **the recovered state is a
prefix of the writes**, with no holes.

In [ ]:
framed_path = os.path.join(WORKDIR, "framed.wal")

w = FramedWal(framed_path)
w.put("user:1", "alice")
w.put("user:2", "bob")
w.close()

# 💥 crash in the middle of writing record #3: header says 40 bytes, only 6 landed.
with open(framed_path, "ab") as f:
    f.write(HEADER.pack(40, 0xDEADBEEF) + b"{\"op\":")

# Restart. Recovery truncates the torn tail before accepting new appends.
w = FramedWal(framed_path)
assert w.truncated_bytes > 0, "recovery should have cut the torn record off"
w.put("user:4", "dana")          # lands at a clean boundary, not glued to garbage
w.put("user:5", "erin")
w.close()

final = FramedWal(framed_path)
print("recovered:", final.data)
final.close()

# The invariant: everything acknowledged is present, nothing is missing from the middle.
assert final.data == {"user:1": "alice", "user:2": "bob",
                      "user:4": "dana",  "user:5": "erin"}, final.data
print("✅ no hole: every record written after the truncation point is intact")

### And a corruption the length prefix alone would miss

Truncation handles a *short* tail. But a crash (or bit-rot) can also leave a record that is
the right length with the wrong bytes. That is what the CRC is for — without it, recovery
would happily feed corrupt data back into the store.

In [ ]:
# Flip one bit inside the payload of the LAST record. Length is untouched.
raw = bytearray(open(framed_path, "rb").read())
raw[-1] ^= 0x01
open(framed_path, "wb").write(raw)

recovered = FramedWal(framed_path)
print("recovered after bit flip:", recovered.data)
recovered.close()

# The corrupt record is rejected and truncated away; the prefix before it is intact.
assert "user:5" not in recovered.data, "a CRC failure must not be replayed"
assert recovered.data == {"user:1": "alice", "user:2": "bob", "user:4": "dana"}
print("✅ corrupt record rejected, earlier records kept — still a clean prefix")

shutil.rmtree(WORKDIR)
print("cleaned up")

## ✅ Recap

The WAL pattern gives us **durability** (data survives crashes), **good write performance** (just appends), and **simple recovery** (just replay).

Real databases (Postgres, SQLite, RocksDB, Kafka…) all use a variant of this idea.

The part people skip when they write their own: **framing**. A WAL is only crash-safe if a
partial write is *detectable*. Length prefix + checksum makes it detectable; recovery then
**truncates** at the first bad record rather than skipping it, so the replayed log is always a
prefix of what the application wrote. Skipping a bad record and carrying on leaves a hole, and
a hole is worse than a short log because nothing downstream can see it.

| | newline-delimited JSON | `[len][crc][payload]` |
|---|---|---|
| Detect a torn tail | only if the JSON happens not to parse | always |
| Detect a torn record followed by more appends | ❌ fuses two records | ✅ length gives the boundary |
| Detect flipped bits in an intact-length record | ❌ | ✅ CRC |
| Recovery result | state with holes | clean prefix |

On top of this core, real engines add **checkpoints** (snapshots so you don't replay forever)
and **log compaction** — that's notebook 3.